SVM Classifier

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
import os

path = 'C:/Users/Aqilah/Downloads/DEGREE/FYP/PROJECT/Coral Image/train'
classes = {'healthy': 0, 'bleached': 1}

In [ ]:
import cv2

X = []
Y = []

for cls, label in classes.items():
    class_path = os.path.join(path, cls)
    for img_file in os.listdir(class_path):
        img_path = os.path.join(class_path, img_file)
        img = cv2.imread(img_path, 0)
        img_resized = cv2.resize(img, (200, 200))
        X.append(img_resized)
        Y.append(label)
X = np.array(X)
Y = np.array(Y)

In [ ]:
#plt.imshow(X[0], cmap='gray')
plt.imshow(X[10])

In [ ]:
X_updated = X.reshape(len(X), -1)
X_updated.shape

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(X_updated, Y, random_state=10, test_size=0.20)
print(xtrain.shape, xtest.shape)

In [ ]:
print(xtrain.max(), xtrain.min())
print(xtest.max(), xtest.min())
xtrain = xtrain/255
xtest = xtest/255
print(xtrain.max(), xtrain.min())
print(xtest.max(), xtest.min())

In [ ]:
from sklearn.decomposition import PCA

print(xtrain.shape, xtest.shape)

pca = PCA(0.90)

pca_train = pca.fit_transform(xtrain)
pca_test = pca.transform(xtest)
print(pca_train.shape, pca_test.shape)

In [ ]:
from sklearn.svm import SVC
import pickle

In [ ]:
C_values = [0.001, 0.01, 0.1, 1]
training_accuracy = []
testing_accuracy = []

for C_value in C_values:
    model = SVC(C=C_value, kernel='rbf', gamma='scale', probability=True)
    model.fit(pca_train, ytrain)
    training_accuracy.append(model.score(pca_train, ytrain))
    testing_accuracy.append(model.score(pca_test, ytest))

In [ ]:
best_C_value = 0.1
final_model = SVC(C=best_C_value, kernel='rbf', gamma='scale', probability=True)
final_model.fit(pca_train, ytrain)

print("Training Score: {:.2f}%".format(final_model.score(pca_train, ytrain) * 100))
print("Testing Score: {:.2f}%".format(final_model.score(pca_test, ytest) * 100))

In [ ]:
# Serialize and save the model to a pickle file
with open('svm.pkl', 'wb') as f:
    pickle.dump(final_model, f)

In [ ]:
# Plot the comparison graph between training and testing accuracy for the best C value
plt.figure(figsize=(8, 6))
plt.plot(C_values, training_accuracy, label='Training Accuracy', marker='o', color='Orange')
plt.plot(C_values, testing_accuracy, label='Testing Accuracy', marker='o', color='b')
plt.axvline(x=best_C_value, color='r', linestyle='--', label='Best C Value')
plt.xscale('log')
plt.xticks(C_values, [str(C_val) for C_val in C_values])
plt.xlabel('C Value (Regularization Parameter)')
plt.ylabel('Accuracy')
plt.title('Training Accuracy vs. Testing Accuracy')
plt.legend()
plt.grid()
plt.show()

In [ ]:
from sklearn.model_selection import learning_curve

# Learning curve function
def plot_learning_curve(estimator, title, X, y, ylim=None, cv=None, n_jobs=None, train_sizes=np.linspace(.1, 1.0, 5)):
    plt.figure(figsize=(8, 6))
    plt.title(title)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.xlabel("Training examples")
    plt.ylabel("Score")
    train_sizes, train_scores, test_scores = learning_curve(estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes)
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    plt.grid()

    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1,
                     color="r")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Cross-validation score")
    plt.legend(loc="best")
    return plt

# Plot learning curve for the final model
title = "Learning Curves (SVM, C={})".format(best_C_value)
cv = 5  # Number of cross-validation folds
plot_learning_curve(final_model, title, pca_train, ytrain, cv=cv, n_jobs=-1)
plt.show()

In [ ]:
pred = model.predict(pca_test)

In [ ]:
from sklearn.metrics import mean_squared_error

print("Testing Score: {:.2f}%".format(final_model.score(pca_test, ytest) * 100))

mse = mean_squared_error(ytest, pred)
print("Mean Squared Error: {:.3f}\n".format(mse))

class_report = classification_report(ytest, pred, target_names=["Healthy", "Bleached"])
print("Classification Report:")
print(class_report)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

prob_pred = final_model.predict_proba(pca_test)[:, 1]

# Assuming ytest contains the true target values (binary, 0 or 1)
auc_roc = roc_auc_score(ytest, prob_pred)
print("AUC-ROC Score: {:.3f}".format(auc_roc))

# Calculate the ROC curve data
fpr, tpr, thresholds = roc_curve(ytest, prob_pred)

# Plot the ROC curve
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='b', label='ROC curve (AUC = {:.3f})'.format(auc_roc))
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
confusion_matrix = confusion_matrix(ytest, pred)

cm_display = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix, display_labels=["Healthy", "Bleached"])
cm_display.plot()
plt.show()

In [ ]:
misclassified = np.where(ytest!=pred)
misclassified
print("Total Misclassified Samples: ", len(misclassified[0]))

In [ ]:
dec = {0: 'Healthy', 1:'Bleached'}

In [ ]:
# Specify the directory path
directory = 'C:/Users/Aqilah/Downloads/DEGREE/FYP/PROJECT/Coral Image/test/healthy'

# Get the list of image files in the directory
image_files = os.listdir(directory)

# Visualize the images
plt.figure(figsize=(12, 11))
for i, image_file in enumerate(image_files[:12]):
    plt.subplot(4, 4, i+1)
    
    # Read the image
    img = cv2.imread(os.path.join(directory, image_file), 0)
    
    # Resize the image
    img_resized = cv2.resize(img, (200, 200))
    
    # Normalize the image
    img_normalized = img_resized/255
    
    # Reshape the image
    img_reshaped = img_normalized.reshape(1, -1)

# Append to a list for batch processing
    if i == 0:
        test_data_pca = pca.transform(img_reshaped)
    else:
        test_data_pca = np.concatenate((test_data_pca, pca.transform(img_reshaped)))
    
# Make predictions for the entire test set at once
predictions = model.predict(test_data_pca)
confidences = model.predict_proba(test_data_pca).max(axis=1)  # Get the maximum probability as confidence

# Map the predicted labels to class names
predicted_classes = [dec[prediction] for prediction in predictions]

# Display the images and predictions
for i, image_file in enumerate(image_files[:12]):
    plt.subplot(4, 4, i+1)
    plt.title(f'Classification: {predicted_classes[i]}\nConfidence: {confidences[i]:.2f}', color='black')
    plt.imshow(cv2.imread(os.path.join(directory, image_file), 0))
    plt.axis('off')
plt.show()

In [ ]:
# Specify the directory path
directory = 'C:/Users/Aqilah/Downloads/DEGREE/FYP/PROJECT/Coral Image/test/bleached'

# Get the list of image files in the directory
image_files = os.listdir(directory)

# Visualize the images
plt.figure(figsize=(12, 10))
for i, image_file in enumerate(image_files[:12]):
    plt.subplot(4, 4, i+1)
    
    # Read the image
    img = cv2.imread(os.path.join(directory, image_file), 0)
    
    # Resize the image
    img_resized = cv2.resize(img, (200, 200))
    
    # Normalize the image
    img_normalized = img_resized / 255
    
    # Reshape the image
    img_reshaped = img_normalized.reshape(1, -1)
    
    # Append to a list for batch processing
    if i == 0:
        test_data_pca = pca.transform(img_reshaped)
    else:
        test_data_pca = np.concatenate((test_data_pca, pca.transform(img_reshaped)))
    
# Make predictions for the entire test set at once
predictions = model.predict(test_data_pca)
confidences = model.predict_proba(test_data_pca).max(axis=1)  # Get the maximum probability as confidence

# Map the predicted labels to class names
predicted_classes = [dec[prediction] for prediction in predictions]

# Display the images and predictions
for i, image_file in enumerate(image_files[:12]):
    plt.subplot(4, 4, i+1)
    plt.title(f'Classification: {predicted_classes[i]}\nConfidence: {confidences[i]:.2f}', color='black')
    plt.imshow(cv2.imread(os.path.join(directory, image_file), 0))
    plt.axis('off')
plt.show()